# Aircraft XML to Excel Preprocessing

This notebook reads aircraft data from an XML file, extracts relevant fields, computes the tire contact area, and exports the results to an Excel file.

In [ ]:
import xml.etree.ElementTree as ET
from openpyxl.styles import Font as XLFont, Alignment
from openpyxl.utils import get_column_letter
import pandas as pd
import numpy as np
from pathlib import Path
import re

xml_path = Path("input_data/aircraft.xml")
tree = ET.parse(xml_path)
root = tree.getroot()

NS_F = "http://schemas.datacontract.org/2004/07/FaarFieldModel"
NS_A = "http://schemas.microsoft.com/2003/10/Serialization/Arrays"
NS_XSI = "http://www.w3.org/2001/XMLSchema-instance"

airplanes = root.find(f".//{{{NS_F}}}Airplanes")
assert airplanes is not None, "Could not find <Airplanes> in the XML."

def get_scalar(parent, tag, default=np.nan, cast=float):
    """Get text of a simple element <tag>value</tag> under parent."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    # handle xsi:nil="true"
    if el.attrib.get(f"{{{NS_XSI}}}nil", "").lower() == "true":
        return default
    if el.text is None:
        return default
    try:
        return cast(el.text.strip())
    except Exception:
        return default

def get_us(parent, tag, default=np.nan, cast=float):
    """Get the <us> child value under a unit-wrapped element <tag><si>..</si><us>..</us></tag>."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    us = el.find(f"{{{NS_F}}}us")
    if us is None or us.text is None:
        return default
    try:
        return cast(us.text.strip())
    except Exception:
        return default

def get_coord_list(parent, tag):
    """Extract (X_us, Y_us) pairs from a list-of-LengthCoordinates element.
    Returns (x_str, y_str) as semicolon-separated US-unit values, or ("", "")."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return "", ""
    xs, ys = [], []
    for child in el:
        x_el = child.find(f"{{{NS_F}}}X/{{{NS_F}}}us")
        y_el = child.find(f"{{{NS_F}}}Y/{{{NS_F}}}us")
        if x_el is not None and x_el.text:
            xs.append(x_el.text.strip())
        if y_el is not None and y_el.text:
            ys.append(y_el.text.strip())
    if not xs:
        return "", ""
    return ";".join(xs), ";".join(ys)

def get_length_list(parent, tag):
    """Extract US values from a list-of-Length element.
    Returns semicolon-separated string, or ""."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return ""
    vals = []
    for child in el:
        us_el = child.find(f"{{{NS_F}}}us")
        if us_el is not None and us_el.text:
            vals.append(us_el.text.strip())
    return ";".join(vals) if vals else ""

In [ ]:
# Tire rules herein are backed by the Bridgestone Aircraft Tire Application Tables and other manufacturer data.
# These are used to assign representative tire sizes/models based on airplane model
#
# Rib profile class assignments follow the Hernandez & Al-Qadi (2015) methodology:
#   - 7-rib: Boeing wide-bodies (validated via ICT-B777-300ER)
#   - 5-rib: Airbus wide-bodies (validated via ICT-A380-800), all narrow-bodies, military transports
#   - 3-rib: GA, business jets, small regional jets/turboprops (tire width < ~11")
#
# Confidence levels:
#   HIGH = validated or manufacturer-inherited from validated reference
#   MED  = cross-referenced from known tire data or size-based classification
#   LOW  = heuristic fallback only

TIRE_RULES = [
    # =========================================================================
    # BOEING — Wide-Body Families → 7-rib (inherited from validated B777-300ER)
    # =========================================================================

    # 747 family
    dict(pattern=r"B?747-8",
         size="52×21.0R22", model="Bridgestone application table (main gear)",
         note="Doc: 747-8 main gear tire size 52×21.0R22", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?747-400\s*ER",
         size="50×20.0R22", model="Bridgestone application table (main gear)",
         note="Doc: 747-400ER main gear tire size 50×20.0R22", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?747-400",
         size="H49.5×19.0-22", model="Bridgestone application table (main gear)",
         note="Doc: 747-400 main gear tire size H49.5×19.0-22", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?747",
         size="H49.5×19.0-22", model="Bridgestone application table (main gear)",
         note="Boeing wide-body inheritance from B777-300ER", ribs=7, confidence="HIGH"),

    # 777 family (VALIDATED via ICT-B777-300ER)
    dict(pattern=r"B?777-?F",
         size="52×21.0R22", model="Bridgestone application table (main gear)",
         note="Doc: 777F main gear tire size 52×21.0R22", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?777-200\s*LR",
         size="52×21.0R22", model="Bridgestone application table (main gear)",
         note="Doc: 777-200LR main gear tire size 52×21.0R22", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?777-300\s*ER",
         size="52×21.0R22", model="Bridgestone application table (main gear)",
         note="Doc: 777-300ER main gear tire size 52×21.0R22 (VALIDATED)", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?777-[89]",
         size="52×21.0R22", model="Bridgestone application table (main gear)",
         note="Doc: 777-8/9 main gear tire size 52×21.0R22", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?777",
         size="50×20.0R22", model="Bridgestone application table (main gear)",
         note="Doc: base 777 main gear tire size 50×20.0R22", ribs=7, confidence="HIGH"),

    # 787 family
    dict(pattern=r"B?787-8\b",
         size="50×20.0R22", model="Bridgestone application table (main gear)",
         note="Doc: 787-8 main gear tire size 50×20.0R22", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?787-(9|10)",
         size="54×21.0R23", model="Bridgestone application table (main gear)",
         note="Doc: 787-9/10 main gear tire size 54×21.0R23", ribs=7, confidence="HIGH"),
    dict(pattern=r"B?787",
         size="50×20.0R22", model="Bridgestone application table (main gear)",
         note="Boeing wide-body inheritance from B777-300ER", ribs=7, confidence="HIGH"),

    # =========================================================================
    # BOEING — Narrow-Body Families → 5-rib
    # =========================================================================

    # 717
    dict(pattern=r"B?717",
         size="H38×12.0-19", model="Cross-reference (narrow-body)",
         note="Narrow-body, ~12in tire width", ribs=5, confidence="MED"),

    # 720
    dict(pattern=r"B?720",
         size="H40×14.5-19", model="Cross-reference (legacy narrow-body)",
         note="Legacy narrow-body", ribs=5, confidence="MED"),

    # 727
    dict(pattern=r"B?727",
         size="H40×14.5-19", model="Bridgestone application table (main gear)",
         note="Narrow-body, H40x14.5-19 class", ribs=5, confidence="MED"),

    # 737 Classics
    dict(pattern=r"B?737-(300|400|500)",
         size="H40×14.5-19", model="Bridgestone application table (main gear)",
         note="Doc: 737-300/400/500 main gear tire size H40×14.5-19", ribs=5, confidence="MED"),
    # 737-600/700 early
    dict(pattern=r"B?737-(600|700)\b",
         size="H43.5×16.0-21", model="Bridgestone application table (main gear)",
         note="Doc: 737-600/700 main gear tire size H43.5×16.0-21", ribs=5, confidence="MED"),
    # 737 NG
    dict(pattern=r"B?737-(800|900)",
         size="H44.5×16.5-21", model="Bridgestone application table (main gear)",
         note="Doc: 737-800/900/900ER main gear tire size H44.5×16.5-21", ribs=5, confidence="MED"),
    # 737 MAX
    dict(pattern=r"B?737\s*MAX|MAX-?(7|8|9|10)\b",
         size="H44.5×16.5R21", model="Bridgestone application table (main gear)",
         note="Doc: 737 MAX main gear tire size H44.5×16.5R21", ribs=5, confidence="MED"),
    # 737 BBJ / generic
    dict(pattern=r"B?737",
         size="H44.5×16.5-21", model="Bridgestone application table (main gear)",
         note="737 family generic, narrow-body", ribs=5, confidence="MED"),

    # 757
    dict(pattern=r"B?757",
         size="H40×14.5-19", model="Bridgestone application table (main gear)",
         note="Doc: 757-200/300 main gear tire size H40×14.5-19", ribs=5, confidence="MED"),

    # 767
    dict(pattern=r"B?767-200\b(?!.*ER)",
         size="H45.5×17.0-20", model="Bridgestone application table (main gear)",
         note="Doc: 767-200 main gear tire size H45.5×17.0-20", ribs=5, confidence="MED"),
    dict(pattern=r"B?767",
         size="H46×18.0-20", model="Bridgestone application table (main gear)",
         note="Doc: 767-200ER/300/400ER main gear; mid-wide but narrow-body tire class", ribs=5, confidence="MED"),

    # =========================================================================
    # AIRBUS — Wide-Body Families → 5-rib (inherited from validated A380)
    # =========================================================================

    # A330
    dict(pattern=r"A330",
         size="1400×530R23", model="Bridgestone application table (main gear)",
         note="Same tire size (1400x530R23) as validated A380", ribs=5, confidence="HIGH"),

    # A340
    dict(pattern=r"A340",
         size="1400×530R23", model="Bridgestone application table (main gear)",
         note="Same tire size (1400x530R23) as validated A380", ribs=5, confidence="HIGH"),

    # A350
    dict(pattern=r"A350",
         size="1400×530R23", model="Bridgestone application table (main gear)",
         note="Airbus wide-body inheritance from A380", ribs=5, confidence="HIGH"),

    # A380 (VALIDATED via ICT-A380-800)
    dict(pattern=r"A380",
         size="1400×530R23", model="Bridgestone application table (main gear)",
         note="Doc: A380-800 main gear tire 1400x530R23 (VALIDATED 5-rib)", ribs=5, confidence="HIGH"),

    # =========================================================================
    # AIRBUS — Narrow-Body Families → 5-rib
    # =========================================================================

    # A220 (formerly CSeries)
    dict(pattern=r"A220|CS-?(100|300)",
         size="", model="Cross-reference (narrow-body)",
         note="Narrow-body (formerly Bombardier CSeries)", ribs=5, confidence="MED"),

    # A300
    dict(pattern=r"A300",
         size="", model="Cross-reference (early wide-body)",
         note="Early wide-body, 12-14in tires, closer to narrow-body tire class", ribs=5, confidence="MED"),

    # A310
    dict(pattern=r"A310",
         size="", model="Cross-reference (mid-range)",
         note="Mid-range, similar to A300", ribs=5, confidence="MED"),

    # A318/A319/A320/A321 family
    dict(pattern=r"A32[01].*NEO|A32[01].*XLR|A319.*NEO",
         size="46×17R20", model="Bridgestone application table (main gear)",
         note="A320 family neo/XLR variants", ribs=5, confidence="MED"),
    dict(pattern=r"A31[89]|A320|A321",
         size="46×17R20", model="Bridgestone application table (main gear)",
         note="Doc: A318-A321 narrow-body family", ribs=5, confidence="MED"),

    # =========================================================================
    # MCDONNELL DOUGLAS → 5-rib
    # =========================================================================

    dict(pattern=r"DC-?8",
         size="", model="Cross-reference (legacy mid-body)",
         note="Legacy mid-body", ribs=5, confidence="MED"),
    dict(pattern=r"DC-?9|MD-?8[0-9]|MD-?90",
         size="", model="Cross-reference (narrow-body class)",
         note="Narrow-body class", ribs=5, confidence="MED"),
    dict(pattern=r"DC-?10|MD-?10|MD-?11",
         size="", model="Cross-reference (wide-body trijet)",
         note="Wide-body trijet, non-Boeing heritage; conservative default", ribs=5, confidence="MED"),

    # =========================================================================
    # OTHER COMMERCIAL / LARGE JETS → 5-rib
    # =========================================================================

    dict(pattern=r"L-?1011|TRISTAR",
         size="", model="Cross-reference (wide-body trijet)",
         note="Wide-body trijet", ribs=5, confidence="MED"),
    dict(pattern=r"CONCORDE",
         size="", model="Cross-reference (supersonic)",
         note="Unique supersonic", ribs=5, confidence="MED"),
    dict(pattern=r"IL-?62|IL-?86",
         size="", model="Cross-reference (Soviet wide-body)",
         note="Soviet wide-bodies", ribs=5, confidence="MED"),
    dict(pattern=r"IL-?76",
         size="", model="Cross-reference (Soviet transport)",
         note="Soviet transport", ribs=5, confidence="MED"),
    dict(pattern=r"TU-?154",
         size="", model="Cross-reference (Soviet mid-range)",
         note="Soviet mid-range jet", ribs=5, confidence="MED"),
    dict(pattern=r"AN-?124",
         size="", model="Cross-reference (heavy transport)",
         note="Heavy transport", ribs=5, confidence="MED"),
    dict(pattern=r"AN-?225",
         size="", model="Cross-reference (heavy transport)",
         note="Heavy transport", ribs=5, confidence="MED"),
    dict(pattern=r"C-?919|COMAC",
         size="", model="Cross-reference (A320-class competitor)",
         note="A320-class competitor", ribs=5, confidence="MED"),
    dict(pattern=r"CV[\s-]?880|CV[\s-]?990",
         size="", model="Cross-reference (legacy jet)",
         note="Legacy jet", ribs=5, confidence="MED"),
    dict(pattern=r"CL[\s-]?44|CANADAIR(?!.*CRJ|.*REGIONAL)",
         size="", model="Cross-reference (legacy jet)",
         note="Canadair CL-44 legacy jet", ribs=5, confidence="MED"),
    dict(pattern=r"CARAVELLE",
         size="", model="Cross-reference (legacy jet)",
         note="Legacy jet", ribs=5, confidence="MED"),
    dict(pattern=r"BAE[\s-]?146|RJ[\s-]?(?:70|85|100)",
         size="", model="Cross-reference (regional jet)",
         note="Regional jet, ~11in tires", ribs=5, confidence="MED"),
    dict(pattern=r"BAC[\s-]?1-?11",
         size="", model="Cross-reference (legacy regional jet)",
         note="Legacy regional jet, ~11.6in tires", ribs=5, confidence="MED"),
    dict(pattern=r"F-?100\b|FOKKER[\s-]?100",
         size="", model="Cross-reference (regional jet)",
         note="Regional jet, ~11in tires", ribs=5, confidence="MED"),
    dict(pattern=r"F-?28\b|FOKKER[\s-]?28",
         size="", model="Cross-reference (regional jet)",
         note="Regional jet, ~11in tires", ribs=5, confidence="MED"),

    # Embraer E-Jets (larger regional jets)
    dict(pattern=r"E-?1[79][05]|E-?JET|EMBRAER[\s-]?1[79][05]",
         size="", model="Cross-reference (regional jet)",
         note="Embraer E-Jet family, 11-16in tires", ribs=5, confidence="MED"),

    # =========================================================================
    # MILITARY → 5-rib (transports/large)
    # =========================================================================

    dict(pattern=r"\bC-?130\b|L-?100",
         size="", model="Cross-reference (military transport)",
         note="Large transport, 16.7in tires", ribs=5, confidence="MED"),
    dict(pattern=r"\bC-?17",
         size="", model="Cross-reference (heavy transport)",
         note="Heavy military transport", ribs=5, confidence="MED"),
    dict(pattern=r"\bC-?5\b",
         size="", model="Cross-reference (heavy transport)",
         note="Heavy military transport", ribs=5, confidence="MED"),
    dict(pattern=r"\bC-?141",
         size="", model="Cross-reference (transport)",
         note="Military transport", ribs=5, confidence="MED"),
    dict(pattern=r"\bC-?123",
         size="", model="Cross-reference (transport)",
         note="Military transport", ribs=5, confidence="MED"),
    dict(pattern=r"A400\s*M",
         size="", model="Cross-reference (transport)",
         note="Military transport, ~12in tires", ribs=5, confidence="MED"),
    dict(pattern=r"B-?52",
         size="", model="Cross-reference (large bomber)",
         note="Large bomber", ribs=5, confidence="MED"),
    dict(pattern=r"KC-?10",
         size="", model="Cross-reference (tanker/transport)",
         note="Same as DC-10 (not Boeing heritage)", ribs=5, confidence="MED"),
    dict(pattern=r"P-?3\b",
         size="", model="Cross-reference (maritime patrol)",
         note="Maritime patrol", ribs=5, confidence="MED"),

    # =========================================================================
    # SMALL AIRCRAFT → 3-rib (tire width < ~11")
    # =========================================================================

    # Small regional jets
    dict(pattern=r"CRJ|CANADAIR[\s-]?REGIONAL",
         size="", model="Cross-reference (small regional jet)",
         note="CRJ family, tires ~8-9in wide", ribs=3, confidence="MED"),
    dict(pattern=r"ERJ[\s-]?(135|140|145)|EMBRAER[\s-]?(135|140|145)",
         size="", model="Cross-reference (small regional jet)",
         note="ERJ family, tires ~7.7in wide", ribs=3, confidence="MED"),

    # Turboprops
    dict(pattern=r"ATR[\s-]?(42|72)",
         size="", model="Cross-reference (turboprop)",
         note="Turboprop, tires ~8-9in wide", ribs=3, confidence="MED"),
    dict(pattern=r"F-?27\b|FOKKER[\s-]?27|F-?50\b|FOKKER[\s-]?50",
         size="", model="Cross-reference (turboprop)",
         note="Fokker turboprop, ~7-10in tires", ribs=3, confidence="MED"),
    dict(pattern=r"Q-?\d{3}|DASH[\s-]?8|DHC-?[78]",
         size="", model="Cross-reference (turboprop)",
         note="De Havilland Canada turboprop family", ribs=3, confidence="MED"),
    dict(pattern=r"SAAB[\s-]?340|SF[\s-]?340",
         size="", model="Cross-reference (turboprop)",
         note="Turboprop, ~10in tires", ribs=3, confidence="MED"),
    dict(pattern=r"SHORT[S]?[\s-]?(330|360)",
         size="", model="Cross-reference (commuter turboprop)",
         note="Commuter turboprop", ribs=3, confidence="MED"),

    # Soviet small
    dict(pattern=r"TU-?134",
         size="", model="Cross-reference (small Soviet jet)",
         note="Small Soviet jet, ~9in tires", ribs=3, confidence="MED"),

    # Legacy propeller
    dict(pattern=r"DC-?3",
         size="", model="Cross-reference (legacy prop)",
         note="Legacy propeller aircraft", ribs=3, confidence="MED"),
    dict(pattern=r"DC-?4",
         size="", model="Cross-reference (legacy prop)",
         note="Legacy propeller aircraft", ribs=3, confidence="MED"),

    # Business jets
    dict(pattern=r"CITATION|CESSNA[\s-]?(?:5\d{2}|6\d{2}|7\d{2}|CITATION)",
         size="", model="Cross-reference (business jet)",
         note="Business jet, small-medium tires", ribs=3, confidence="MED"),
    dict(pattern=r"LEARJET|LEAR[\s-]?\d",
         size="", model="Cross-reference (business jet)",
         note="Business jet, small tires (~5.75in)", ribs=3, confidence="MED"),
    dict(pattern=r"GULFSTREAM|G-?[IV]{1,3}\b|G-?[2345]\d{2}\b|GV\b|GIV\b",
         size="", model="Cross-reference (business jet)",
         note="Business jet, small-medium tires", ribs=3, confidence="MED"),
    dict(pattern=r"FALCON|DASSAULT",
         size="", model="Cross-reference (business jet)",
         note="Business jet, small tires (~5.75in)", ribs=3, confidence="MED"),
    dict(pattern=r"HAWKER|HS[\s-]?125|BAE[\s-]?125",
         size="", model="Cross-reference (business jet)",
         note="Business jet, small tires (~5.75in)", ribs=3, confidence="MED"),
    dict(pattern=r"SABRELINER|T-?39",
         size="", model="Cross-reference (small military jet)",
         note="Small military/business jet, ~5in tires", ribs=3, confidence="MED"),
    dict(pattern=r"BEECHJET|PREMIER",
         size="", model="Cross-reference (business jet)",
         note="Business jet, small tires", ribs=3, confidence="MED"),
    dict(pattern=r"CHALLENGER|CL-?60[0-9]",
         size="", model="Cross-reference (business jet)",
         note="Business jet, small-medium tires", ribs=3, confidence="MED"),

    # General aviation
    dict(pattern=r"CESSNA|CARAVAN",
         size="", model="Cross-reference (GA)",
         note="General aviation, small tires (w=4-9in)", ribs=3, confidence="MED"),
    dict(pattern=r"PIPER|CHEROKEE|NAVAJO|SENECA|CHEYENNE",
         size="", model="Cross-reference (GA)",
         note="General aviation, small tires", ribs=3, confidence="MED"),
    dict(pattern=r"BEECH(?:CRAFT)?|BONANZA|BARON|KING[\s-]?AIR",
         size="", model="Cross-reference (GA/turboprop)",
         note="GA/turboprop, small tires (w=4-9in)", ribs=3, confidence="MED"),
    dict(pattern=r"PILATUS|PC-?12",
         size="", model="Cross-reference (turboprop)",
         note="Single turboprop, small tires", ribs=3, confidence="MED"),

    # Military fighters
    dict(pattern=r"F-?15\b",
         size="", model="Cross-reference (fighter)",
         note="Fighter, single-wheel, small-medium tires", ribs=3, confidence="MED"),
    dict(pattern=r"F-?16\b",
         size="", model="Cross-reference (fighter)",
         note="Fighter, single-wheel, small tires (~8.75in)", ribs=3, confidence="MED"),
    dict(pattern=r"F[\s/-]?18|F[\s/-]?A[\s-]?18",
         size="", model="Cross-reference (fighter)",
         note="Fighter, single-wheel, small tires (~8.8in)", ribs=3, confidence="MED"),

    # Non-airplane vehicles
    dict(pattern=r"TRUCK|ARFF|FIRE",
         size="", model="Cross-reference (vehicle)",
         note="Commercial vehicle tires", ribs=3, confidence="MED"),
]


RIB_PROFILES = {
    7: {
        "Ribs (mm)": "50;35;40;90;40;35;50",
        "Load Factor": "0.24;0.08;0.08;0.20;0.08;0.08;0.24",
        "Stress Factor": "1.80;1.10;1.10;1.10;1.10;1.10;1.80",
    },
    5: {
        "Ribs (mm)": "70;45;90;45;70",
        "Load Factor": "0.28;0.11;0.22;0.11;0.28",
        "Stress Factor": "1.80;1.10;1.10;1.10;1.80",
    },
    3: {
        "Ribs (mm)": "70;90;70",
        "Load Factor": "0.35;0.30;0.35",
        "Stress Factor": "1.80;1.10;1.80",
    },
}

# Heuristic threshold: area-only, for generic/reference aircraft with no real identity
AREA_3_MAX = 150.0  # in² — boundary between regional/commuter and narrow-body jets


def _to_float_or_nan(x):
    try:
        v = float(x)
        if np.isnan(v) or v <= 0:
            return np.nan
        return v
    except Exception:
        return np.nan

def rib_fallback(area_in2):
    """
    Area-only heuristic for generic/reference aircraft.
    Returns: (ribs_class, confidence, rationale)
    Never assigns 7-rib (requires validated cross-reference data).
    """
    area = _to_float_or_nan(area_in2)

    if pd.isna(area):
        return 5, "LOW", "No area data → default profile class = 5"

    if area < AREA_3_MAX:
        return 3, "LOW", f"Area {area:.1f} in² < {AREA_3_MAX} → 3-rib"
    else:
        return 5, "LOW", f"Area {area:.1f} in² ≥ {AREA_3_MAX} → 5-rib"

# ---------- RULE MATCHING HARDENING ----------

def norm_name(name: str) -> str:
    s = (name or "").upper()
    s = s.replace("–", "-").replace("−", "-")
    # normalize separators to spaces, keep alphanumerics and hyphen
    s = re.sub(r"[^A-Z0-9\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Precompile once (do this right after defining TIRE_RULES)
for rule in TIRE_RULES:
    rule["rx"] = re.compile(rule["pattern"], flags=re.IGNORECASE)

def assign_rep_tire_and_ribs(airplane_name, area_in2, width_in):
    s = norm_name(airplane_name)

    for rule in TIRE_RULES:
        if rule["rx"].search(s):
            conf = rule.get("confidence", "MED")
            note = f"{rule['note']} | ribs={rule['ribs']} (profile class)"
            return rule["size"], rule["model"], note, rule["ribs"], "FamilyRule", conf

    ribs, conf, why = rib_fallback(area_in2)
    note = f"Heuristic ({conf}): {why}"
    return "", "", note, ribs, "Heuristic", conf

In [ ]:
rows = []
for ap in list(airplanes):
    # only keep the AirplaneInfo blocks
    if ap.tag != f"{{{NS_A}}}anyType":
        continue
    if ap.attrib.get(f"{{{NS_XSI}}}type") != "AirplaneInfo":
        continue

    name = ap.find(f"{{{NS_F}}}Name")
    name = name.text.strip() if (name is not None and name.text) else ""

    # Requested fields
    gw_lbs = get_us(ap, "_GrossWeight")          # "Gross Taxi Weight (lbs)"
    tire_area = get_us(ap, "TireArea", default=0.0)
    tire_len = get_us(ap, "TireLength", default=0.0)
    tire_wid = get_us(ap, "TireWidth", default=0.0)

    # TirePressureF is often nil; default to 0 per your requirement
    tire_pressure = get_us(ap, "Cp", default=0.0)
    mg_percent = get_scalar(ap, "MgPercent")
    mg_percent_pcn = get_scalar(ap, "MgPercentPCN")

    num_gear = get_scalar(ap, "NumberGear", cast=int)
    num_tracks = get_scalar(ap, "NumberTireTracks", cast=int, default=1.0)
    num_wheels = get_scalar(ap, "NumberWheels", cast=int)

    # New gear geometry fields
    manufacturer    = get_scalar(ap, "Manufacturer", default="", cast=str)
    deprecated      = get_scalar(ap, "Deprecated", default="false", cast=str)
    gear            = get_scalar(ap, "Gear", default="", cast=str)
    gear_orient     = get_scalar(ap, "GearOrientation", default=0, cast=int)
    tt              = get_us(ap, "Tt", default=0.0)
    b_spacing       = get_us(ap, "B", default=0.0)
    ts              = get_scalar(ap, "Ts", default=0.0)
    tg              = get_scalar(ap, "Tg", default=0.0)
    tv              = get_scalar(ap, "Tv", default=0.0)
    wc_x, wc_y     = get_coord_list(ap, "WheelCoordinates")
    ep_x, ep_y      = get_coord_list(ap, "EvaluationPoints")
    tire_track_x    = get_length_list(ap, "TireTrackX")

    rows.append({
        "Airplane Name": name,
        "Manufacturer": manufacturer,
        "Deprecated": deprecated,
        "Gross Taxi Weight (lbs)": gw_lbs,
        "Tire Pressure (psi)": tire_pressure,
        "Percent GW on Gear": mg_percent,
        "MgPercentPCN": mg_percent_pcn,
        "Number Gear": num_gear,
        "Number Tire Tracks": num_tracks,
        "Number Wheels": num_wheels,
        "Gear": gear,
        "GearOrientation": gear_orient,
        "Tt (in.)": tt,
        "B (in.)": b_spacing,
        "Ts (in.)": ts,
        "Tg (in.)": tg,
        "Tv (in.)": tv,
        "Tire Contact Width (in.)": tire_wid,
        "Tire Contact Length (in.)": tire_len,
        "Tire Contact Area (in.^2)": tire_area,
        "WheelCoord_X (in.)": wc_x,
        "WheelCoord_Y (in.)": wc_y,
        "EvalPt_X (in.)": ep_x,
        "EvalPt_Y (in.)": ep_y,
        "TireTrackX (in.)": tire_track_x,
    })

df = pd.DataFrame(rows)

df["Number Tire Tracks"] = df["Number Tire Tracks"].replace(np.nan, 1) # Use 1 as default, to avoid division by 0 later

# Manual Entries
df = pd.concat([
    df,
    pd.DataFrame([{
        "Airplane Name": "ICT-B777-300ER",
        "Manufacturer": "Boeing",
        "Deprecated": "false",
        "Gross Taxi Weight (lbs)": 777000,
        "Tire Pressure (psi)": 200.0,
        "Percent GW on Gear": 0.525,
        "MgPercentPCN": 0.2660,
        "Number Gear": 3,
        "Number Tire Tracks": 4,
        "Number Wheels": 6,
        "Gear": "X",
        "GearOrientation": 90,
        "Tt (in.)": 0.0,
        "B (in.)": 0.0,
        "Ts (in.)": 0.0,
        "Tg (in.)": 0.0,
        "Tv (in.)": 0.0,
        "Tire Contact Width (in.)": 13.39,
        "Tire Contact Length (in.)": 14.96,
        "Tire Contact Area (in.^2)": 200.0,
        "WheelCoord_X (in.)": "-243.5;-188.5;-243.5;-188.5;-243.5;-188.5;243.5;188.5;243.5;188.5;243.5;188.5",
        "WheelCoord_Y (in.)": "-57.6;-57.6;0;0;57.6;57.6;57.6;57.6;0;0;-57.6;-57.6",
        "EvalPt_X (in.)": "-188.5;-194;-199.5;-205;-210.5;-216",
        "EvalPt_Y (in.)": "0;0;0;0;0;0",
        "TireTrackX (in.)": "-243.5;-188.5;188.5;243.5",
    }, {
        "Airplane Name": "ICT-A380-800",
        "Manufacturer": "Airbus",
        "Deprecated": "false",
        "Gross Taxi Weight (lbs)": 1239000,
        "Tire Pressure (psi)": 210.3,
        "Percent GW on Gear": 0.38,
        "MgPercentPCN": 0.19,
        "Number Gear": 3,
        "Number Tire Tracks": 4,
        "Number Wheels": 4,
        "Gear": "X",
        "GearOrientation": 90,
        "Tt (in.)": 0.0,
        "B (in.)": 0.0,
        "Ts (in.)": 0.0,
        "Tg (in.)": 0.0,
        "Tv (in.)": 0.0,
        "Tire Contact Width (in.)": 14.17,
        "Tire Contact Length (in.)": 22.05,
        "Tire Contact Area (in.^2)": 270.00,
        "WheelCoord_X (in.)": "-218.66999816894531;-271.76999816894534;-271.76999816894534;-218.66999816894531;218.66999816894531;271.76999816894534;271.76999816894534;218.66999816894531",
        "WheelCoord_Y (in.)": "95.55;162.45;95.55;162.45;95.55;162.45;95.55;162.45",
        "EvalPt_X (in.)": "-218.64519576894531;-223.95999816894531;-229.2749981689453;-234.5899981689453;-239.90499816894533;-245.21999816894532;-245.21999816894532;-245.21999816894532",
        "EvalPt_Y (in.)": "95.55;97.98;100.4;102.83;105.56;107.69;118.35;129",
        "TireTrackX (in.)": "-271.77;-218.67;218.67;271.77",
    }])
], ignore_index=True)


df["Load (lbs)"] = (df["Gross Taxi Weight (lbs)"] * df["MgPercentPCN"]) / df["Number Wheels"]
df["Load (N)"] = df["Load (lbs)"] * 4.44822
df["Tire Pressure (MPa)"] = df["Tire Pressure (psi)"] * 0.00689476
df["Tire Contact Length (mm)"] = df["Tire Contact Length (in.)"] * 25.4
df["Tire Contact Width (mm)"] = df["Tire Contact Width (in.)"] * 25.4
df["Tire Contact Area (cm.^2)"] = df["Tire Contact Area (in.^2)"] * 6.4516
# Optional: round the contact geometry to match your example output style
df["Tire Contact Width (in.)"] = df["Tire Contact Width (in.)"].round(1)
df["Tire Contact Length (in.)"] = df["Tire Contact Length (in.)"].round(1)
df["Tire Contact Area (in.^2)"] = df["Tire Contact Area (in.^2)"].round(1)


#=====
# Number of Ribs Assigned
#=====
rep = df.apply(
    lambda r: assign_rep_tire_and_ribs(
        r.get("Airplane Name", ""),
        r.get("Tire Contact Area (in.^2)", np.nan),
        r.get("Tire Contact Width (in.)", np.nan)
    ),
    axis=1,
    result_type="expand"
)

rep.columns = [
    "Repr. Tire Size",
    "Repr. Tire Model",
    "Repr. Model SourceNote",
    "Est_NRibs",
    "Rib_Method",
    "Rib_Confidence"
]

insert_at = df.columns.get_loc("Tire Contact Area (in.^2)") + 1
for col in rep.columns[::-1]:
    df.insert(insert_at, col, rep[col])
    
df["Est_NRibs"] = pd.to_numeric(df["Est_NRibs"], errors="coerce").fillna(5).astype(int)
#=====
#=====

# Categorization
df["Ribs (mm)"] = df["Est_NRibs"].map(lambda n: RIB_PROFILES.get(n, RIB_PROFILES[5])["Ribs (mm)"])
df["Load Factor"] = df["Est_NRibs"].map(lambda n: RIB_PROFILES.get(n, RIB_PROFILES[5])["Load Factor"])
df["Stress Factor"] = df["Est_NRibs"].map(lambda n: RIB_PROFILES.get(n, RIB_PROFILES[5])["Stress Factor"])


num_formats = {
    "Gross Taxi Weight (lbs)": "0",
    "Tire Pressure (psi)": "0.0",
    "Percent GW on Gear": "0.0000",
    "MgPercentPCN": "0.0000",
    "Number Gear": "0",
    "Number Tire Tracks": "0",
    "Number Wheels": "0",
    "GearOrientation": "0",
    "Tt (in.)": "0.00",
    "B (in.)": "0.00",
    "Ts (in.)": "0.00",
    "Tg (in.)": "0.00",
    "Tv (in.)": "0.00",
    "Tire Contact Width (in.)": "0.0",
    "Tire Contact Length (in.)": "0.0",
    "Tire Contact Area (in.^2)": "0.00",
    "Load (lbs)": "0",
    "Load (N)": "0",
    "Tire Pressure (MPa)": "0.000",
    "Tire Contact Length (mm)": "0",
    "Tire Contact Width (mm)": "0",
    "Tire Contact Area (cm.^2)": "0.00",
    "Est_NRibs": "0"
}

for c in num_formats:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")


out_path = Path("input_data/aircraft.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Aircraft", index=False)

    ws = writer.sheets["Aircraft"]

    base_font    = XLFont(name="Times New Roman", size=11)
    header_font  = XLFont(name="Times New Roman", size=11, bold=True)
    center_align = Alignment(horizontal="center", vertical="center")

    # Font + alignment
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row,
                            min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = header_font if cell.row == 1 else base_font
            cell.alignment = center_align

    # Map headers -> column index
    header_to_col = {
        ws.cell(row=1, column=col).value: col
        for col in range(1, ws.max_column + 1)
    }

    # Apply numeric formats (and coerce stray strings to numbers)
    for col_name, fmt in num_formats.items():
        col_idx = header_to_col.get(col_name)
        if not col_idx:
            continue
        for r in range(2, ws.max_row + 1):
            cell = ws.cell(row=r, column=col_idx)
            if isinstance(cell.value, str):
                try:
                    cell.value = float(cell.value)
                except Exception:
                    pass
            cell.number_format = fmt

    # Auto-fit widths
    for col_idx in range(1, ws.max_column + 1):
        max_len = 0
        for r in range(1, ws.max_row + 1):
            v = ws.cell(row=r, column=col_idx).value
            if v is None:
                continue
            if isinstance(v, float) and pd.isna(v):
                continue
            max_len = max(max_len, len(str(v)))
        ws.column_dimensions[get_column_letter(col_idx)].width = max_len + 2

In [11]:
# Visualize the first rows of the dataframe
df.head()

,Airplane Name,Gross Taxi Weight (lbs),Tire Pressure (psi),Percent GW on Gear,MgPercentPCN,Number Gear,Number Tire Tracks,Number Wheels,Tire Contact Width (in.),Tire Contact Length (in.),...,Rib_Confidence,Load (lbs),Load (N),Tire Pressure (MPa),Tire Contact Length (mm),Tire Contact Width (mm),Tire Contact Area (cm.^2),Ribs (mm),Load Factor,Stress Factor
0,SWL-2,2000.0,30.0,1.0,1.0,1,1,1,7.3,11.7,...,HIGH,2000.0,8896.44,0.206843,296.007901,185.004938,430.106650,70;90;70,0.35;0.30;0.35,1.80;1.10;1.80
1,SWL-5,5000.0,45.0,1.0,1.0,1,1,1,9.4,15.0,...,LOW,5000.0,22241.10,0.310264,382.144581,238.840363,716.844466,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
2,SWL-10,10000.0,50.0,1.0,1.0,1,1,1,12.6,20.2,...,HIGH,10000.0,44482.20,0.344738,512.700760,320.437975,1290.320000,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
3,Single Wheel 2,2000.0,30.0,1.0,0.5,1,1,1,0.0,0.0,...,LOW,1000.0,4448.22,0.206843,0.000000,0.000000,0.000000,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
4,Single Wheel 5,5000.0,45.0,1.0,0.5,1,1,1,0.0,0.0,...,LOW,2500.0,11120.55,0.310264,0.000000,0.000000,0.000000,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
